In [1]:
# ===================== CORE LIBRARIES =====================
import numpy as np                      # Numbers and arrays, fast math on columns of data
import pandas as pd                     # Load CSV files and work with tables (DataFrames)
import matplotlib.pyplot as plt         # Basic charts: bar, line, histogram
import seaborn as sns                   # Nicer statistical charts (heatmaps, boxplots)
import joblib                           # Save and load trained models (.joblib files)
import warnings                         # Controls warning messages
warnings.filterwarnings("ignore")       # Hide warnings to keep the output clean

# ============ PREPROCESSING AND MODEL SELECTION ============
from sklearn.model_selection import (
    train_test_split,                   # Split data into training and testing parts
    StratifiedKFold,                    # Cross-validation that keeps class proportions equal in every fold
    cross_val_score,                    # Score a model across several folds
    GridSearchCV)                       # Try many hyperparameter combinations to find the best

from sklearn.preprocessing import (
    StandardScaler,                     # Scale numbers to mean 0 and spread 1 (helps SVM, KNN, MLP)
    OneHotEncoder,                      # Turn categories (e.g. city) into 0/1 columns
    LabelEncoder)                       # Turn text labels (Low/Moderate/High) into numbers (0/1/2)

from sklearn.compose import ColumnTransformer  # Apply different preprocessing to number and text columns
from sklearn.pipeline import Pipeline          # Chain preprocessing and model into one object
from sklearn.impute import SimpleImputer       # Fill missing values (median or most frequent)

# ======================== ML MODELS ========================
from sklearn.linear_model import LogisticRegression      # Simple, fast, easy-to-explain baseline
from sklearn.neighbors import KNeighborsClassifier       # Predicts from the most similar patients
from sklearn.svm import SVC                              # Finds the best boundary between classes
from sklearn.tree import DecisionTreeClassifier          # If-then rules, easy to read
from sklearn.ensemble import (
    RandomForestClassifier,             # Many decision trees voting together, strong and stable
    GradientBoostingClassifier,         # Trees built one after another, each fixing earlier errors
    AdaBoostClassifier)                 # Boosting that focuses on hard-to-classify rows
from sklearn.naive_bayes import GaussianNB               # Fast probability-based model
from xgboost import XGBClassifier                        # Powerful, popular boosted trees
from lightgbm import LGBMClassifier                      # Fast boosted trees, good on large data

# ========================== METRICS ==========================
from sklearn.metrics import (
    accuracy_score,                     # Share of predictions that are correct
    f1_score,                           # Balance of precision and recall (use macro for 3 classes)
    classification_report,              # Precision, recall and F1 for each class
    confusion_matrix,                   # Table of correct and wrong predictions per class
    ConfusionMatrixDisplay)             # Draws the confusion matrix as a chart

# ============ IMBALANCED DATA AND EXPLANATION ============
from imblearn.over_sampling import SMOTE   # Creates synthetic samples of rare classes (e.g. High risk)
import shap                                # Explains why the model made each prediction

# ================= DEEP LEARNING (KERAS) =================
import tensorflow as tf                    # Main deep learning library from Google
from tensorflow import keras               # Easy interface for building neural networks
from tensorflow.keras import layers, models  # layers = building blocks, models = the network itself

# ============ DEEP LEARNING (PYTORCH, ALTERNATIVE) ============
import torch                               # Tensors (like numpy arrays, but GPU-capable)
import torch.nn as nn                      # Neural network layers and loss functions
import torch.optim as optim                # Optimizers that update weights (Adam, SGD)

# ======================== FRONTEND ========================
import streamlit as st                     # Turns your Python script into a web app

## Data Loading & Initial Inspection

In [2]:
df = pd.read_csv('diabetes_risk.csv')
df.head()

,patient_id,age,gender,city,bmi,family_history_diabetes,physical_activity_level,diet_type,smoking_status,alcohol_consumption,hours_sleep_per_night,stress_level,fasting_blood_sugar,hba1c_level,blood_pressure_systolic,blood_pressure_diastolic,waist_circumference_cm,income_bracket,diabetes_risk
0,1,47,Female,Mumbai,26.5,Yes,Sedentary,Vegetarian,Never,Never,9.0,8,178,7.3,146,90,107.1,High,Low
1,2,53,Female,Mumbai,20.5,Yes,Moderate,Vegetarian,Never,Never,7.0,6,152,6.9,156,100,93.8,Low,Low
2,3,47,Male,Thane,31.3,No,Moderate,Non-Vegetarian,Current,Regular,7.0,7,168,6.6,150,102,113.5,Low,High
3,4,41,Male,Bengaluru,22.8,No,Sedentary,Non-Vegetarian,Never,Never,9.0,5,155,6.7,126,92,109.6,Middle,Low
4,5,57,Female,Bengaluru,16.7,No,Sedentary,Non-Vegetarian,Current,Occasional,6.5,10,188,6.7,130,90,80.8,High,Low


In [3]:
df.shape

(15000, 19)

In [4]:
df.columns

Index(['patient_id', 'age', 'gender', 'city', 'bmi', 'family_history_diabetes',
       'physical_activity_level', 'diet_type', 'smoking_status',
       'alcohol_consumption', 'hours_sleep_per_night', 'stress_level',
       'fasting_blood_sugar', 'hba1c_level', 'blood_pressure_systolic',
       'blood_pressure_diastolic', 'waist_circumference_cm', 'income_bracket',
       'diabetes_risk'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   patient_id                15000 non-null  int64  
 1   age                       15000 non-null  int64  
 2   gender                    15000 non-null  object 
 3   city                      15000 non-null  object 
 4   bmi                       15000 non-null  float64
 5   family_history_diabetes   15000 non-null  object 
 6   physical_activity_level   15000 non-null  object 
 7   diet_type                 15000 non-null  object 
 8   smoking_status            14528 non-null  object 
 9   alcohol_consumption       11212 non-null  object 
 10  hours_sleep_per_night     15000 non-null  float64
 11  stress_level              15000 non-null  int64  
 12  fasting_blood_sugar       15000 non-null  int64  
 13  hba1c_level               15000 non-null  float64
 14  blood_